# Strategy Workbench Colab Runner

이 노트북은 여러 전략을 한 번에 비교하고, 기간/이평선/분할매도/트레일링 스탑 같은 조건을 바꿔가며 백테스트할 수 있도록 만든 실행기입니다.

사용 방법:

1. 아래 설정 셀에서 `OUTPUT_NAME`과 `CONFIG`를 원하는 값으로 바꿉니다.
2. 셀을 위에서부터 순서대로 실행합니다.
3. `metrics.csv`와 equity curve 차트를 확인합니다.
4. 마지막 셀에서 결과 파일 묶음을 zip으로 내려받을 수 있습니다.

주의:

- 복잡한 전략 비교 도구라서, 일반 폼 입력보다 `CONFIG` 딕셔너리를 직접 수정하는 방식으로 두었습니다.
- 같은 형식으로 전략 블록을 복사하면 전략을 여러 개 동시에 비교할 수 있습니다.
- GitHub에 올리기 전에는 아래 `REPO_URL`을 실제 저장소 주소로 바꿔야 합니다.


In [ ]:
REPO_URL = "https://github.com/YOUR_NAME/YOUR_REPO.git"  # change this
BRANCH = "main"


In [ ]:
from pathlib import Path
import shutil
import subprocess

if "YOUR_NAME" in REPO_URL:
    raise ValueError("Please update REPO_URL before running the notebook.")

repo_dir = Path('/content/strategy_workbench_repo')
if repo_dir.exists():
    shutil.rmtree(repo_dir)

subprocess.run([
    'git', 'clone', '--depth', '1', '-b', BRANCH, REPO_URL, str(repo_dir)
], check=True)

print(f'Repo directory: {repo_dir}')


In [ ]:
OUTPUT_NAME = 'strategy_workbench_demo'

CONFIG = {
    'start_date': '1985-10-01',
    'end_date': '2026-03-09',
    'commission_rate': 0.001,
    'tax_rate': 0.22,
    'expense_ratio': 0.0095,
    'borrow_spread': 1.0,
    'strategies': [
        {
            'name': 'Price vs 200 SMA | 3x',
            'base_series': 'ndx',
            'leverage': 3.0,
            'include_financing_cost': True,
            'exit_destination': 'cash',
            'entry': {
                'type': 'price_above_sma',
                'source': 'traded',
                'window': 200,
                'confirm_days': 3,
            },
            'exit': {
                'type': 'price_below_sma',
                'source': 'traded',
                'window': 200,
                'confirm_days': 1,
            },
            'take_profit_ladder': [],
            'trailing_stop': {
                'enabled': False,
            },
        },
        {
            'name': '3-161-185 + Partial Exits | 3x',
            'base_series': 'ndx',
            'leverage': 3.0,
            'include_financing_cost': True,
            'exit_destination': 'cash',
            'entry': {
                'type': 'all_of',
                'rules': [
                    {
                        'type': 'price_above_sma',
                        'source': 'traded',
                        'window': 200,
                        'confirm_days': 3,
                    },
                    {
                        'type': 'sma_chain_above',
                        'source': 'traded',
                        'windows': [3, 161, 185],
                        'confirm_days': 1,
                    },
                ],
            },
            'exit': {
                'type': 'any_of',
                'rules': [
                    {
                        'type': 'price_below_sma',
                        'source': 'traded',
                        'window': 200,
                        'confirm_days': 1,
                    },
                    {
                        'type': 'sma_chain_below',
                        'source': 'traded',
                        'windows': [3, 161, 185],
                        'confirm_days': 1,
                    },
                ],
            },
            'take_profit_ladder': [
                {
                    'trigger_gain_multiple': 1.15,
                    'sell_fraction': 0.50,
                    'destination': 'cash',
                },
                {
                    'trigger_gain_multiple': 1.68,
                    'sell_fraction': 0.35,
                    'destination': 'spy',
                },
            ],
            'trailing_stop': {
                'enabled': True,
                'activate_after_first_take_profit': True,
                'drawdown_from_peak': 0.15,
                'destination': 'cash',
            },
        },
    ],
}

print('Current strategies:')
for strategy in CONFIG['strategies']:
    print('-', strategy['name'])


In [ ]:
import json
import subprocess
from pathlib import Path

config_path = Path('/content/workbench_config.json')
config_path.write_text(json.dumps(CONFIG, indent=2), encoding='utf-8')

subprocess.run(
    ['python', 'run_workbench.py', '--config', str(config_path), '--output-name', OUTPUT_NAME],
    cwd=repo_dir,
    check=True,
)

print(f'Run complete: {OUTPUT_NAME}')


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

output_dir = Path('/root/strategy_workbench_outputs') / OUTPUT_NAME
metrics = pd.read_csv(output_dir / 'metrics.csv')
display(metrics)

curves = pd.read_csv(output_dir / 'equity_curves.csv')
pivot = curves.pivot(index='date', columns='strategy', values='equity')
pivot.plot(figsize=(12, 6), title='Strategy Comparison')
plt.ylabel('Equity Multiple')
plt.show()


In [ ]:
!cd /root && zip -r /content/strategy_workbench_outputs.zip strategy_workbench_outputs

from google.colab import files
files.download('/content/strategy_workbench_outputs.zip')
